# VASU V3 — binary DS-CNN with hard-negative mining

This notebook uses **Speaker 1–3 train/development data only**. Speaker 4 and the
phone-recorded Speaker 5 remain sealed. It trains from scratch; no pretrained
keyword model or proprietary SDK is used.

Run **Runtime → Run all**. When Cell 1 asks, select these two files together:

- `recordings.zip`
- `vasu_dataset_audit_checkpoint_v1.zip`

There are three code cells. Cell 2 may take several minutes. Keep the tab open.
At the end, Colab downloads one results ZIP. Do not upload any Speaker 5 file.

The model input is 1.2 seconds / 59×40×1. The development set is used for
selection, so its numbers are not independent test accuracy.

## Cell 1 — upload, verify and prepare features

In [ ]:
import os, sys, json, io, zipfile, hashlib, re, shutil, time, importlib.util
from pathlib import Path
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '2')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '2')
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.fft import rfft
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 1729
SMOKE = os.environ.get('VASU_SMOKE', '0') == '1'  # Local engineering check only.
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass
BASE = Path(os.environ.get('VASU_BASE', '/content'))
RUN = BASE / ('vasu_v3_smoke' if SMOKE else 'vasu_training_v3')
RUN.mkdir(parents=True, exist_ok=True)
SR = 16000
# Preserve the complete 1.2-second context that was manually reviewed.
# This deliberately replaces the OLD 1-second / 49-frame frontend.
N = 19200
FRAME, HOP, FFT, MELS = 480, 320, 512, 40
FRAMES = 1 + (N - FRAME) // HOP  # 59
LABELS = ['vasu', 'confusable', 'other_speech', 'background']
EPOCHS = 2 if SMOKE else 32
THRESHOLDS = np.array([.35, .45, .55, .65, .75, .85, .90, .94, .97, .99])
print('TensorFlow:', tf.__version__, '| Python:', sys.version.split()[0])
print('GPU:', bool(tf.config.list_physical_devices('GPU')))
print('Model input:', (FRAMES, MELS, 1), '| duration: 1.2 seconds')
print('Binary VASU training. Speakers 4 and 5 sealed. No pretrained KWS model.')
assert FRAMES == 59

required_zips = ['recordings.zip', 'vasu_dataset_audit_checkpoint_v1.zip']
if not all((BASE / f).exists() for f in required_zips):
    from google.colab import files
    print('Select BOTH recordings.zip and vasu_dataset_audit_checkpoint_v1.zip')
    files.upload()
assert all((BASE / f).exists() for f in required_zips), 'Both ZIP files are required.'

def read_csv(z, name):
    matches = [p for p in z.namelist() if Path(p).name == name]
    assert len(matches) == 1, f'Ambiguous/missing checkpoint file: {name}'
    return pd.read_csv(z.open(matches[0])).fillna('')

with zipfile.ZipFile(BASE / required_zips[1]) as z:
    manifest = read_csv(z, 'vasu_dataset_manifest_v1.csv')
    words = read_csv(z, 'accepted_word_candidates_v1.csv')
    rejected = read_csv(z, 'rejected_word_candidates_v1.csv')
    negatives = read_csv(z, 'negative_windows_final_v1.csv')
assert len(words) == 415 and len(negatives) == 1197
assert not words.speaker.eq('speaker04').any()
assert not negatives.speaker.eq('speaker04').any()
active = manifest[manifest.split.isin(['train', 'development'])].copy()
assert not active.speaker.eq('speaker04').any()
assert active.filename.is_unique
assert manifest.groupby('sha256').split.nunique().max() == 1
sources = active.set_index('filename').to_dict('index')
audio = {}
with zipfile.ZipFile(BASE / required_zips[0]) as z:
    names = z.namelist()
    for row in active.itertuples():
        matches = [p for p in names if Path(p).name == row.filename]
        assert len(matches) == 1, f'Missing/duplicate WAV: {row.filename}'
        raw = z.read(matches[0])  # No speaker04 bytes are read.
        assert hashlib.sha256(raw).hexdigest() == row.sha256, f'Changed WAV: {row.filename}'
        rate, x = wavfile.read(io.BytesIO(raw))
        assert rate == SR and x.ndim == 1 and x.dtype == np.int16
        audio[row.filename] = x

def bounds(filename, include_rejected=True):
    result = []
    text = sources[filename]['exclusions']
    for a, b in re.findall(r'(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)', str(text)):
        result.append((max(0, float(a) - .25), float(b) + .25))
    # Previously declared manual exclusions from the long-candidate review.
    extra = {
        'spk02_dev_kw_mixed_s05.wav': [(22.3, 25.9)],
        'spk03_train_confusable_s01.wav': [(83.8, 86.5)],
        'spk02_train_kw_normal_s01.wav': [(45.8, 46.9)],
    }
    result.extend(extra.get(filename, []))
    if include_rejected:
        for row in rejected[rejected.filename.eq(filename)].itertuples():
            result.append((float(row.review_start_s), float(row.review_end_s)))
    return result

def safe_interval(filename, start, end):
    return not any(start < b and end > a for a, b in bounds(filename))

def extract(filename, start):
    x = audio[filename]
    a = int(round(start * SR))
    assert a >= 0 and a + N <= len(x), (filename, start)
    return x[a:a + N].astype(np.float32) / 32768.0

items, omitted = [], []
for row in words.itertuples():
    start = float(row.review_start_s)
    end = start + N / SR
    if abs(float(row.review_end_s) - end) >= .002:
        omitted.append({'filename': row.filename, 'start_s': start, 'reason': 'short_boundary_review_clip', 'id': row.candidate_id})
        continue
    if start < 0 or end > len(audio[row.filename]) / SR or not safe_interval(row.filename, start, end):
        omitted.append({'filename': row.filename, 'start_s': start, 'reason': 'boundary_or_exclusion_overlap', 'id': row.candidate_id})
        continue
    items.append({'filename': row.filename, 'speaker': row.speaker, 'split': row.split,
                  'label': row.proposed_label, 'start_s': start, 'end_s': end, 'id': row.candidate_id})
for row in negatives.itertuples():
    # Extend reviewed 1s negative windows by 0.1s on each side using ORIGINAL audio.
    # Clamp to recording boundaries, then recheck exclusions over the ENTIRE new window.
    duration = len(audio[row.filename]) / SR
    start = max(0., min(float(row.start_s) - .1, duration - N / SR))
    end = start + N / SR
    if not safe_interval(row.filename, start, end):
        omitted.append({'filename': row.filename, 'start_s': start, 'reason': 'expanded_negative_exclusion', 'id': row.window_id})
        continue
    items.append({'filename': row.filename, 'speaker': row.speaker, 'split': row.split,
                  'label': row.model_label, 'start_s': start, 'end_s': end, 'id': row.window_id})
items = pd.DataFrame(items).drop_duplicates(['filename', 'start_s', 'label']).reset_index(drop=True)
assert items.groupby('filename').split.nunique().max() == 1
assert not items.speaker.eq('speaker04').any()
items.to_csv(RUN / 'used_examples.csv', index=False)
pd.DataFrame(omitted).to_csv(RUN / 'excluded_context_windows.csv', index=False)
print('Loaded', len(audio), 'train/development recordings; 0 test recordings.')
print(items.groupby(['split', 'label']).size().to_string())
print('Context/exclusion checks removed:', len(omitted), '(logged, not silently relabeled)')
print('No additional human listening is required for this preparation step.')

# PCM -> per-frame mean removal -> periodic Hann -> RFFT -> power -> HTK Mel -> ln.
# No audio-library defaults, no per-clip peak normalization, no test-derived statistics.
window = (0.5 - 0.5 * np.cos(2 * np.pi * np.arange(FRAME) / FRAME)).astype(np.float32)
hz_to_mel = lambda f: 2595.0 * np.log10(1 + f / 700.)
mel_to_hz = lambda m: 700. * (10 ** (m / 2595.) - 1)
edges = mel_to_hz(np.linspace(hz_to_mel(80.), hz_to_mel(7600.), MELS + 2))
freq = np.arange(FFT // 2 + 1) * SR / FFT
mel = np.maximum(0., np.minimum((freq[:, None] - edges[:-2]) / (edges[1:-1] - edges[:-2]),
                              (edges[2:] - freq[:, None]) / (edges[2:] - edges[1:-1]))).astype(np.float32)

def frontend(x):
    x = np.asarray(x, np.float32)
    assert x.shape[-1] == N
    f = np.lib.stride_tricks.sliding_window_view(x, FRAME, axis=-1)[..., ::HOP, :]
    f = (f - f.mean(axis=-1, keepdims=True)) * window
    spectrum = rfft(f, n=FFT, axis=-1)
    power = (spectrum.real ** 2 + spectrum.imag ** 2) / float(FFT * FFT)
    return np.log(np.maximum(power @ mel, 1e-12)).astype(np.float32)

rng = np.random.default_rng(SEED)
train_items = items[items.split.eq('train')].reset_index(drop=True)
dev_items = items[items.split.eq('development')].reset_index(drop=True)
noise_sources = [f for f, r in sources.items() if r['split'] == 'train' and r['recording_category'] == 'background']
assert noise_sources

def augment(x):
    y = x.copy()
    # Only shift when the samples that would be dropped are quiet relative to the clip.
    shift = int(rng.integers(-1600, 1601))
    if shift:
        edge = y[-shift:] if shift > 0 else y[:-shift]
        if np.sqrt(np.mean(edge ** 2) + 1e-12) < .15 * np.sqrt(np.mean(y ** 2) + 1e-12):
            fill = np.zeros_like(y)
            if shift > 0:
                fill[shift:] = y[:-shift]
            else:
                fill[:shift] = y[-shift:]
            y = fill
    y *= 10 ** (rng.uniform(-6., 4.) / 20.)
    if rng.random() < .65:
        source = audio[rng.choice(noise_sources)].astype(np.float32) / 32768.
        start = int(rng.integers(0, len(source) - N + 1))
        n = source[start:start + N].copy()
        n -= n.mean()
        signal_rms = np.sqrt(np.mean(y ** 2) + 1e-12)
        noise_rms = np.sqrt(np.mean(n ** 2) + 1e-12)
        y += n * signal_rms / (noise_rms * 10 ** (rng.uniform(12., 30.) / 20.))
    peak = np.max(np.abs(y))
    if peak > .98:
        y *= .98 / peak
    return y

train_wave = np.stack([extract(r.filename, r.start_s) for r in train_items.itertuples()])
dev_wave = np.stack([extract(r.filename, r.start_s) for r in dev_items.itertuples()])
base_features = frontend(train_wave)
mean = base_features.mean(axis=(0, 1)).astype(np.float32)
std = np.maximum(base_features.std(axis=(0, 1)), .5).astype(np.float32)
def normalize(f):
    return np.clip((f - mean) / std, -8., 8.).astype(np.float32)[..., None]

X_train_parts = [normalize(base_features)]
for repetition in range(1 if SMOKE else 4):
    X_train_parts.append(normalize(frontend(np.stack([augment(x) for x in train_wave]))))
X_train = np.concatenate(X_train_parts)
X_dev = normalize(frontend(dev_wave))
y_base = np.array([LABELS.index(x) for x in train_items.label], dtype=np.int32)
y_train = np.tile(y_base, len(X_train_parts))
y_dev = np.array([LABELS.index(x) for x in dev_items.label], dtype=np.int32)
# Equal total weight per class, then per represented speaker within a class.
weights = np.empty(len(train_items), np.float32)
for label, group in train_items.groupby('label'):
    counts = group.speaker.value_counts()
    for idx, row in group.iterrows():
        weights[idx] = 1. / (len(LABELS) * len(counts) * counts[row.speaker])
weights /= weights.mean()
sample_weights = np.tile(weights, len(X_train_parts))
np.savez(RUN / 'frontend_constants.npz', hann=window, mel=mel, mean=mean, std=std)
frontend_config = dict(sample_rate=SR, pcm_scale=32768., audio_samples=N, input_shape=[FRAMES, MELS, 1],
                      frame_samples=FRAME, hop_samples=HOP, fft_size=FFT, mel_bins=MELS,
                      mel_low_hz=80., mel_high_hz=7600., mel_formula='HTK; triangles on FFT-bin frequencies; no area normalization',
                      window='periodic Hann', dc_removal='per-frame mean, before Hann',
                      power_divisor=FFT*FFT, log='natural', log_floor=1e-12,
                      normalization='training-only per-Mel mean/std; clip [-8,8]',
                      frame_padding=False, labels=LABELS, inference_hop_samples=1600)
(RUN / 'frontend.json').write_text(json.dumps(frontend_config, indent=2))
assert X_train.shape[1:] == (59, 40, 1) and np.isfinite(X_train).all()
print('Original training windows:', len(train_items), '| including variations:', len(X_train))
print('Development windows:', len(X_dev), '| shape:', X_dev.shape[1:])
print('Variations are NOT extra speakers or independent recordings.')

## Cell 2 — train, mine hard negatives and create INT8 model

In [ ]:

# Binary labels: Vasu=1; every reviewed non-keyword category=0.
Y_train = (y_train == 0).astype(np.float32)
Y_dev = (y_dev == 0).astype(np.float32)
THRESHOLDS_V3 = np.array([.30, .40, .50, .60, .70, .80, .90, .95, .98], np.float32)

def balanced_weights(y):
    y = np.asarray(y).astype(bool)
    w = np.empty(len(y), np.float32)
    w[y] = .5 / max(1, y.sum())
    w[~y] = .5 / max(1, (~y).sum())
    return w / w.mean()

W_train = balanced_weights(Y_train)
W_dev = balanced_weights(Y_dev)

regularizer = tf.keras.regularizers.l2(2e-4)

def ds_block(x, channels, stride=1):
    x = tf.keras.layers.DepthwiseConv2D(
        3, strides=stride, padding='same', use_bias=False,
        depthwise_regularizer=regularizer)(x)
    x = tf.keras.layers.BatchNormalization(momentum=.9)(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Conv2D(
        channels, 1, use_bias=False,
        kernel_regularizer=regularizer)(x)
    x = tf.keras.layers.BatchNormalization(momentum=.9)(x)
    return tf.keras.layers.ReLU()(x)

def build_binary_dscnn(kind='temporal'):
    inp = tf.keras.Input((FRAMES, MELS, 1))
    x = tf.keras.layers.Conv2D(
        12, (5, 3), strides=2, padding='same', use_bias=False,
        kernel_regularizer=regularizer)(inp)
    x = tf.keras.layers.BatchNormalization(momentum=.9)(x)
    x = tf.keras.layers.ReLU()(x)
    x = ds_block(x, 20, 2)
    x = ds_block(x, 28, 2)
    if kind == 'temporal':
        # Pool frequency but preserve the time order needed for Vasu/Kaasu.
        x = tf.keras.layers.AveragePooling2D((1, int(x.shape[2])))(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dropout(.35)(x)
    x = tf.keras.layers.Dense(
        12, activation='relu', kernel_regularizer=regularizer)(x)
    out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    return tf.keras.Model(inp, out, name='vasu_binary_' + kind)

def model_scores(model, X, batch=128):
    return np.concatenate([
        model(X[i:i+batch], training=False).numpy().reshape(-1)
        for i in range(0, len(X), batch)
    ])

def clip_metrics(scores, threshold):
    detected = scores >= threshold
    positive = Y_dev.astype(bool)
    recalls = []
    for speaker in ('speaker01', 'speaker02', 'speaker03'):
        mask = positive & dev_items.speaker.eq(speaker).to_numpy()
        if mask.any(): recalls.append(float(detected[mask].mean()))
    negative_rates = {}
    for label in ('confusable', 'other_speech', 'background'):
        mask = dev_items.label.eq(label).to_numpy()
        negative_rates[label] = float(detected[mask].mean()) if mask.any() else 0.
    return dict(
        recall=float(detected[positive].mean()),
        worst_speaker_recall=min(recalls),
        confusable_false=negative_rates['confusable'],
        other_false=negative_rates['other_speech'],
        background_false=negative_rates['background'])

def clip_rank(metrics):
    penalty = (3*metrics['confusable_false'] +
               metrics['other_false'] + metrics['background_false']) / 5
    if metrics['recall'] >= .90 and metrics['worst_speaker_recall'] >= .80:
        return (0, penalty, -metrics['recall'])
    return (1, -(.65*metrics['recall'] +
                  .35*metrics['worst_speaker_recall'] - penalty), penalty)

class DetectorSnapshots(tf.keras.callbacks.Callback):
    def __init__(self, name, keep=2):
        super().__init__()
        self.name, self.keep = name, keep
        self.saved, self.rows = [], []
    def on_epoch_end(self, epoch, logs=None):
        scores = model_scores(self.model, X_dev)
        choices = []
        for threshold in THRESHOLDS_V3:
            metrics = clip_metrics(scores, float(threshold))
            choices.append((clip_rank(metrics), float(threshold), metrics))
        rank, threshold, metrics = min(choices, key=lambda x: x[0])
        row = dict(model=self.name, epoch=epoch+1, threshold=threshold,
                   parameters=int(self.model.count_params()), **metrics,
                   loss=float(logs['loss']), val_loss=float(logs['val_loss']))
        self.rows.append(row)
        path = RUN / f'{self.name}_epoch_{epoch+1:02d}.keras'
        if len(self.saved) < self.keep or rank < max(x[0] for x in self.saved):
            self.model.save(path)
            self.saved.append((rank, str(path), row))
            self.saved.sort(key=lambda x: x[0])
            self.saved = self.saved[:self.keep]
        print(f"  {self.name} epoch {epoch+1}: recall={metrics['recall']:.1%}, "
              f"confusable false={metrics['confusable_false']:.1%}, threshold={threshold:.2f}")

def train_candidate(kind, name, X, y, weights, epochs):
    tf.keras.utils.set_random_seed(SEED)
    model = build_binary_dscnn(kind)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(.001, clipnorm=1.),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=.02))
    snapshots = DetectorSnapshots(name)
    start = time.perf_counter()
    model.fit(
        X, y, sample_weight=weights,
        validation_data=(X_dev, Y_dev, W_dev),
        batch_size=32, epochs=epochs, shuffle=True, verbose=0,
        callbacks=[
            snapshots,
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', patience=3, factor=.5, min_lr=2e-5),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=7)])
    print(name, 'parameters:', model.count_params(),
          '| seconds:', round(time.perf_counter()-start, 1))
    return snapshots

# Pass 1 exists only to find difficult TRAINING negatives.
print('\nPASS 1/2 — initial binary DS-CNN for training-only hard-negative mining')
base_snapshots = train_candidate(
    'temporal', 'mining_model', X_train, Y_train, W_train,
    2 if SMOKE else 18)
base_path = base_snapshots.saved[0][1]
mining_model = tf.keras.models.load_model(base_path, compile=False)

# Scan only training recordings whose declared category is non-keyword.
eligible_categories = {'confusable', 'other_speech', 'mixed_nonkeyword', 'background'}
scan_sources = active[
    active.split.eq('train') &
    active.recording_category.isin(eligible_categories)
].copy()
scan_rows = []
for source in scan_sources.itertuples():
    duration = len(audio[source.filename]) / SR
    starts = np.arange(0., duration - N/SR + 1e-7, .10)
    starts = np.array([
        s for s in starts
        if safe_interval(source.filename, float(s), float(s)+N/SR)
    ])
    if not len(starts): continue
    for offset in range(0, len(starts), 128):
        batch_starts = starts[offset:offset+128]
        wave = np.stack([extract(source.filename, float(s)) for s in batch_starts])
        scores = model_scores(mining_model, normalize(frontend(wave)))
        for start_s, score in zip(batch_starts, scores):
            scan_rows.append(dict(
                filename=source.filename,
                speaker=source.speaker,
                source_category=source.recording_category,
                start_s=float(start_s), end_s=float(start_s)+N/SR,
                mining_score=float(score)))

scan = pd.DataFrame(scan_rows).sort_values('mining_score', ascending=False)
selected_hard = []
per_file = {}
for row in scan.itertuples():
    previous = per_file.setdefault(row.filename, [])
    if len(previous) >= (4 if SMOKE else 30): continue
    if any(abs(row.start_s - old) < .60 for old in previous): continue
    previous.append(row.start_s)
    selected_hard.append(row._asdict())
    if len(selected_hard) >= (24 if SMOKE else 300): break
hard_manifest = pd.DataFrame(selected_hard)
assert len(hard_manifest) > 0
hard_manifest.to_csv(RUN / 'training_hard_negatives.csv', index=False)
hard_wave = np.stack([
    extract(row.filename, row.start_s)
    for row in hard_manifest.itertuples()
])
X_hard = normalize(frontend(hard_wave))
X_hard_aug = normalize(frontend(np.stack([augment(x) for x in hard_wave])))
X_final = np.concatenate([X_train, X_hard, X_hard_aug])
Y_final = np.concatenate([
    Y_train,
    np.zeros(len(X_hard) + len(X_hard_aug), np.float32)
])
W_final = balanced_weights(Y_final)
print('Mined hard negatives:', len(hard_manifest))
print('Final training examples including variations:', len(X_final))
print('Speaker4/5 examples used:', 0)

print('\nPASS 2/2 — final bounded comparison')
all_snapshots, training_rows = [], []
for kind in ('temporal', 'dense'):
    snapshots = train_candidate(
        kind, 'final_' + kind, X_final, Y_final, W_final,
        2 if SMOKE else 30)
    all_snapshots.extend(snapshots.saved)
    training_rows.extend(snapshots.rows)
pd.DataFrame(training_rows).to_csv(RUN / 'training_history.csv', index=False)

# Continuous development streams at two phase alignments.
stream_features, stream_meta = [], []
for source in active[active.split.eq('development')].itertuples():
    for alignment in (0., .05):
        duration = len(audio[source.filename]) / SR
        starts = np.arange(alignment, duration-N/SR+1e-7, .10)
        starts = np.array([
            s for s in starts
            if safe_interval(source.filename, float(s), float(s)+N/SR)
        ])
        if not len(starts): continue
        first = len(stream_features)
        for offset in range(0, len(starts), 128):
            wave = np.stack([
                extract(source.filename, float(s))
                for s in starts[offset:offset+128]
            ])
            stream_features.extend(normalize(frontend(wave)))
        accepted = words[
            words.filename.eq(source.filename) &
            words.proposed_label.eq('vasu')
        ]
        ids_used = set(dev_items.id)
        expected = [
            (float(row.review_start_s), float(row.review_end_s), row.candidate_id)
            for row in accepted.itertuples()
            if row.candidate_id in ids_used
        ]
        stream_meta.append(dict(
            filename=source.filename,
            category=source.recording_category,
            speaker=source.speaker,
            alignment=float(alignment), starts=starts,
            expected=expected, begin=first, end=len(stream_features)))
X_stream = np.asarray(stream_features, np.float32)

def trigger_events(scores, starts, threshold, confirmations):
    events, run, last_event, last_start = [], 0, -1e9, -1e9
    for score, start in zip(scores, starts):
        if start-last_start > .101: run = 0
        run = run+1 if score >= threshold else 0
        event_time = float(start+N/SR)
        if run >= confirmations and event_time-last_event >= 1.0:
            events.append(event_time)
            last_event, run = event_time, 0
        last_start = start
    return events

def continuous_metrics(scores, threshold, confirmations):
    rows = []
    for ref in stream_meta:
        events = trigger_events(
            scores[ref['begin']:ref['end']], ref['starts'],
            threshold, confirmations)
        used, matched = set(), 0
        for event in events:
            choices = [
                i for i, (a, b, _) in enumerate(ref['expected'])
                if i not in used and a <= event <= b+.50
            ]
            if choices:
                used.add(choices[0]); matched += 1
        rows.append(dict(
            filename=ref['filename'], category=ref['category'],
            speaker=ref['speaker'], alignment=ref['alignment'],
            matched=matched, expected=len(ref['expected']),
            events=len(events), unmatched=max(0, len(events)-matched),
            monitored_s=len(ref['starts'])*.10))
    rows = pd.DataFrame(rows)
    results = []
    for alignment, group in rows.groupby('alignment'):
        positive = group[group.category.eq('keyword')]
        negative = group[~group.category.eq('keyword')]
        per_speaker = positive.groupby('speaker')[['matched','expected']].sum()
        recalls = per_speaker.matched / per_speaker.expected.clip(lower=1)
        results.append(dict(
            recall=float(positive.matched.sum()/max(1, positive.expected.sum())),
            worst_speaker_recall=float(recalls.min()),
            false_events=int(negative.events.sum()),
            false_events_per_min=float(
                60*negative.events.sum()/max(.1, negative.monitored_s.sum()))))
    return dict(
        recall=min(r['recall'] for r in results),
        worst_speaker_recall=min(r['worst_speaker_recall'] for r in results),
        false_events=max(r['false_events'] for r in results),
        false_events_per_min=max(r['false_events_per_min'] for r in results)), rows

def continuous_rank(metrics, parameters):
    if metrics['recall'] >= .90 and metrics['worst_speaker_recall'] >= .80:
        return (0, metrics['false_events_per_min'], -metrics['recall'], parameters)
    return (1, -metrics['recall']+.04*metrics['false_events_per_min'],
            -metrics['worst_speaker_recall'], parameters)

search_rows, winner = [], None
for _, path, checkpoint in all_snapshots:
    candidate = tf.keras.models.load_model(path, compile=False)
    scores = model_scores(candidate, X_stream)
    for confirmations in (1, 2, 3):
        for threshold in THRESHOLDS_V3:
            metrics, details = continuous_metrics(
                scores, float(threshold), confirmations)
            row = dict(
                path=path, model=candidate.name,
                epoch=checkpoint['epoch'], threshold=float(threshold),
                confirmations=confirmations,
                parameters=int(candidate.count_params()), **metrics)
            search_rows.append(row)
            rank = continuous_rank(metrics, candidate.count_params())
            if winner is None or rank < winner['rank']:
                winner = dict(rank=rank, row=row, details=details)

search = pd.DataFrame(search_rows)
search.to_csv(RUN / 'continuous_development_search.csv', index=False)
selected = tf.keras.models.load_model(winner['row']['path'], compile=False)
selected.save(RUN / 'selected_float.keras')
winner['details'].to_csv(RUN / 'float_development_by_recording.csv', index=False)
print('\nSELECTED FLOAT DETECTOR')
print(winner['row'])

# Full signed-INT8 conversion with fixed batch 1.
cal_rng = np.random.default_rng(SEED)
calibration = []
for label in (0., 1.):
    indices = np.flatnonzero(Y_final == label)
    calibration.extend(
        cal_rng.choice(indices, min(150, len(indices)), replace=False))
def representative():
    for index in calibration:
        yield [X_final[index:index+1].astype(np.float32)]
fixed_input = tf.keras.Input(batch_shape=(1, FRAMES, MELS, 1))
fixed_model = tf.keras.models.clone_model(selected, input_tensors=fixed_input)
fixed_model.set_weights(selected.get_weights())
converter = tf.lite.TFLiteConverter.from_keras_model(fixed_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
blob = converter.convert()
(RUN / 'vasu_binary_int8.tflite').write_bytes(blob)
interpreter = tf.lite.Interpreter(model_content=blob, num_threads=1)
interpreter.allocate_tensors()
input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]
assert input_info['dtype'] == np.int8 and output_info['dtype'] == np.int8
assert not any(
    np.issubdtype(tensor['dtype'], np.floating)
    for tensor in interpreter.get_tensor_details())
input_scale, input_zero = input_info['quantization']
output_scale, output_zero = output_info['quantization']
def quantize_feature(x):
    return np.clip(
        np.rint(x/input_scale+input_zero), -128, 127).astype(np.int8)
def int8_scores(X):
    result = []
    for x in X:
        interpreter.set_tensor(input_info['index'], quantize_feature(x[None]))
        interpreter.invoke()
        q = interpreter.get_tensor(output_info['index']).reshape(-1)[0]
        result.append((float(q)-output_zero)*output_scale)
    return np.asarray(result, np.float32)

int8_stream = int8_scores(X_stream)
int8_choices = []
for confirmations in (1, 2, 3):
    for threshold in THRESHOLDS_V3:
        metrics, details = continuous_metrics(
            int8_stream, float(threshold), confirmations)
        rank = continuous_rank(metrics, selected.count_params())
        int8_choices.append((rank, float(threshold), confirmations, metrics, details))
_, final_threshold, final_confirmations, final_metrics, final_details = min(
    int8_choices, key=lambda x: x[0])
final_details.to_csv(RUN / 'int8_development_by_recording.csv', index=False)
pd.DataFrame([
    dict(threshold=t, confirmations=c, **m)
    for _, t, c, m, _ in int8_choices
]).to_csv(RUN / 'int8_detector_search.csv', index=False)

ops = sorted(set(
    operation['op_name'] for operation in interpreter._get_ops_details()
) - {'DELEGATE'})
summary = dict(
    status='development_selected_not_independently_tested',
    approach='binary DS-CNN plus training-only hard-negative mining',
    model=selected.name,
    parameters=int(selected.count_params()),
    model_bytes=len(blob),
    model_sha256=hashlib.sha256(blob).hexdigest(),
    threshold=final_threshold,
    confirmations=final_confirmations,
    cooldown_seconds=1.0,
    inference_hop_seconds=.10,
    input_scale=float(input_scale), input_zero=int(input_zero),
    output_scale=float(output_scale), output_zero=int(output_zero),
    int8_trigger_code=int(np.ceil(
        final_threshold/output_scale+output_zero)),
    operators=ops,
    int8_development=final_metrics,
    development_target=dict(
        recall_at_least=.90,
        every_speaker_recall_at_least=.80,
        hard_negative_false_events_per_min_below=1.0),
    development_target_met=bool(
        final_metrics['recall'] >= .90 and
        final_metrics['worst_speaker_recall'] >= .80 and
        final_metrics['false_events_per_min'] < 1.0),
    speaker4_used=False, speaker5_used=False,
    hardware_ram_bytes=None, hardware_cpu_percent=None,
    hardware_inference_ms=None, asr_handoff_latency_ms=None)
(RUN / 'result_summary.json').write_text(json.dumps(summary, indent=2))
print('\nINT8 DEVELOPMENT RESULT')
print(json.dumps(summary, indent=2))
print('Speaker 4 and Speaker 5 remain sealed.')


## Cell 3 — export evidence and ESP handoff files

In [ ]:

# Export ESP handoff files, plots and one downloadable ZIP.
def c_array(name, values, ctype='float'):
    flat = np.asarray(values).reshape(-1)
    def format_value(value):
        if ctype != 'float': return str(int(value))
        text = format(float(value), '.9g')
        if '.' not in text and 'e' not in text: text += '.0'
        return text + 'f'
    rows = [
        ', '.join(format_value(v) for v in flat[i:i+12])
        for i in range(0, len(flat), 12)]
    return (f'static const {ctype} {name}[{len(flat)}] = {{\n' +
            ',\n'.join(rows) + '\n};\n')

header = '#pragma once\n#include <stdint.h>\n'
header += f'#define VASU_SAMPLE_RATE {SR}\n'
header += f'#define VASU_AUDIO_SAMPLES {N}\n'
header += f'#define VASU_FRAMES {FRAMES}\n#define VASU_MELS {MELS}\n'
header += f'#define VASU_INT8_TRIGGER {summary["int8_trigger_code"]}\n'
header += f'#define VASU_CONFIRMATIONS {final_confirmations}\n'
header += c_array('vasu_hann', window)
header += c_array('vasu_mel_bin_major', mel)
header += c_array('vasu_feature_mean', mean)
header += c_array('vasu_feature_std', std)
(RUN / 'vasu_frontend_constants.h').write_text(header)

model_hex = [
    ', '.join(f'0x{byte:02x}' for byte in blob[i:i+16])
    for i in range(0, len(blob), 16)]
(RUN / 'vasu_model_data.cc').write_text(
    '#include <stdint.h>\n'
    'alignas(16) extern const unsigned char g_vasu_model[] = {\n' +
    ',\n'.join(model_hex) +
    '\n};\nextern const unsigned int g_vasu_model_len = sizeof(g_vasu_model);\n')
(RUN / 'vasu_model_data.h').write_text(
    '#pragma once\nextern const unsigned char g_vasu_model[];\n'
    'extern const unsigned int g_vasu_model_len;\n')

# Four reference inputs: Vasu, confusable, other speech and background.
gold_indices = [int(np.flatnonzero(y_base == label)[0]) for label in range(4)]
gold_pcm = np.rint(train_wave[gold_indices]*32768.).astype(np.int16)
gold_features = normalize(frontend(gold_pcm.astype(np.float32)/32768.))
gold_inputs = quantize_feature(gold_features)
gold_outputs = int8_scores(gold_features)
np.savez(
    RUN / 'golden_vectors.npz', pcm=gold_pcm,
    features=gold_features, quantized_input=gold_inputs,
    output=gold_outputs, source_class=np.arange(4))
gold_pcm.astype('<i2').tofile(RUN / 'golden_pcm_s16le.bin')
gold_features.astype('<f4').tofile(RUN / 'golden_features_f32le.bin')
gold_inputs.tofile(RUN / 'golden_inputs_int8.bin')

history = pd.read_csv(RUN / 'training_history.csv')
figure, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
colors = {'final_temporal':'#176b9b', 'final_dense':'#bd7621'}
for name, group in history.groupby('model'):
    axes[0].plot(group.epoch, group.loss, color=colors[name], label=name+' train')
    axes[0].plot(group.epoch, group.val_loss, '--', color=colors[name], label=name+' dev')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Binary DS-CNN training')
axes[0].legend(fontsize=8)
tradeoff = pd.read_csv(RUN / 'int8_detector_search.csv')
for confirmations, group in tradeoff.groupby('confirmations'):
    axes[1].plot(group.false_events_per_min, group.recall,
                 marker='o', label=f'{confirmations} confirmation(s)')
axes[1].axhline(.90, color='#555555', linestyle=':', label='90% recall target')
axes[1].set(xlabel='False events/min on hard development negatives',
            ylabel='Reviewed keyword recall',
            title='INT8 detector trade-off')
axes[1].legend(fontsize=8)
figure.savefig(RUN / 'development_results.png', dpi=150)
plt.show()

(RUN / 'READ_ME_FIRST.txt').write_text(
    'VASU V3 uses speakers 1-3 only. Speakers 4 and 5 remain sealed.\n'
    'Results are development-selected, not independent accuracy.\n'
    'Input: 1.2 s / 19200 samples / 59x40x1 log-Mel features.\n'
    'Do not combine with the old 1-second/49x40 firmware.\n'
    'Do not open a sealed test set unless development performance is accepted.\n'
    'Hardware RAM, CPU and latency remain unmeasured until ESP deployment.\n')
(RUN / 'environment.json').write_text(json.dumps(
    dict(python=sys.version, tensorflow=tf.__version__,
         numpy=np.__version__, seed=SEED), indent=2))

archive = Path(shutil.make_archive(str(RUN)+'_results', 'zip', RUN))
print('RESULT ZIP:', archive)
print('Download it and send result_summary.json or the entire ZIP here.')
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    pass
